# TOPS Technologies - Data Analytics Program
## Module 7-10 Assessment: NumPy, Pandas, Matplotlib, Seaborn (Code: M10-A1)
---
This notebook provides the complete, production-grade solutions for all sections:
- **Section A:** Concept Application (Technical Explanations & Scenarios S1-S6)
- **Section B:** Practical Coding Tasks (Tasks 1 to 4)
- **Section C:** Mini Capstone Project (Food Delivery Analytics Console)
- **Section D:** AI-Augmented Learning (Prompts, Bug Analysis, and Corrections)


---
## SECTION A: CONCEPT APPLICATION

### Scenario 1: Delivery Fee Calculator (NumPy Vectorisation vs For-Loop)
**1. Why NumPy Vectorisation is Technically Superior:**
- **Contiguous Memory & C-Level Execution:** A NumPy array stores homogeneous numeric data in a contiguous block of memory, allowing operations to execute via optimized, pre-compiled C/Fortran vector instructions (SIMD - Single Instruction, Multiple Data). A standard Python `for-loop` operates in interpreted bytecode with dynamic type checking, boxing, and unboxing overhead on every single iteration across the 50,000 items, resulting in execution times that are 50x to 100x slower.
- **Broadcasting with Scalar Rates:** When applying a scalar fee rate (e.g., `Rs 5 * distance_array`), NumPy automatically broadcasts the scalar value `5` across the entire 1D array of shape `(50000,)` without replicating the scalar in memory. The scalar-vector multiplication occurs entirely in a low-level vectorised loop.
- **When a Python Loop is Still Necessary:** A loop is necessary when calculations involve **stateful serial dependencies**, where each element's value strictly depends on the output of the immediately preceding element (e.g., an iterative Markov chain simulation, state-machine tracking, or complex recursive cumulative charging logic that cannot be expressed as a linear prefix scan).

### Scenario 2: Restaurant Performance Reporting (Pandas `groupby().agg()` vs For-Loop)
**1. Why `groupby().agg()` is the Correct Approach:**
- Filtering inside an iterative loop performs repeated table scans ($O(N \times K)$ complexity, where $N$ is rows and $K$ is unique restaurants) and repeatedly allocates intermediary DataFrame slices. Conversely, `groupby()` partitions the DataFrame using optimized hash tables in Cython ($O(N)$ complexity) and computes aggregations in a single vectorised pass.
**2. Specific Method Chain:**
```python
restaurant_summary = (
    df.groupby('restaurant_name')
    .agg(
        avg_order_value=('order_value', 'mean'),
        avg_delivery_time=('delivery_time_mins', 'mean'),
        avg_rating=('rating', 'mean')
    )
)
```
**3. Resulting DataFrame Index:**
- The resulting DataFrame's index represents the **unique grouping keys** (the unique `restaurant_name` entries). Each row index uniquely labels an aggregated summary record for that specific restaurant brand.

### Scenario 3: Monthly Performance Report (Matplotlib Chart Selection & Layout)
**1. Chart Type Selection & Justification:**
- **Metric (a) Total Orders Placed per Month:** **Line Chart (`plt.plot`)**. Total orders over 12 months is continuous, chronological time-series data; a line chart highlights seasonal trends, trajectories, and month-over-month rate of change.
- **Metric (b) Proportion of Orders by Cuisine Type:** **Bar Chart or Donut/Pie Chart (`plt.bar` or `plt.pie`)**. Categorical composition across four discrete cuisine types is best visualized with a discrete bar chart or donut chart to compare relative shares without distortion.
- **Metric (c) Distribution of Delivery Times:** **Histogram or KDE Plot (`plt.hist`)**. Delivery time is a continuous numeric variable; a histogram bins the values to illustrate skewness, central tendency, modal frequency, and spread.
**2. Configuration of `plt.subplots()`:**
- Arrange in a 1-row by 3-column layout (`fig, axes = plt.subplots(1, 3, figsize=(18, 5))`) or 3-row by 1-column layout for vertical reports.
**3. Shared X-Axis Suitability:**
- A shared x-axis (`sharex=True`) **would NOT be appropriate**. The three subplots have completely incompatible x-axis domains: Subplot 1 uses chronological months (Jan-Dec), Subplot 2 uses categorical labels (Cuisines), and Subplot 3 uses a continuous numerical scale (Delivery Time in minutes). Sharing an axis would corrupt or misalign the scales.

### Scenario 4: Peak Identification with `plt.annotate()`
**1. Programmatic Implementation:**
```python
max_idx = np.argmax(daily_orders)
max_day = days[max_idx]
max_volume = daily_orders[max_idx]

plt.annotate(
    f'Peak Day {max_day}: {max_volume:,} orders',
    xy=(max_day, max_volume),
    xytext=(max_day + 5, max_volume + 150),
    arrowprops=dict(facecolor='crimson', shrink=0.05, width=1.5, headwidth=8),
    fontsize=10, fontweight='bold', color='crimson'
)
```
**2. Parameters Explained:**
- `xy`: A tuple `(x, y)` defining the exact point on the chart that the arrow points to (the coordinates of the maximum spike).
- `xytext`: A tuple `(x, y)` defining the position where the text label is rendered.
- `arrowprops`: A dictionary of display properties (color, head width, shrink) controlling the visual appearance and geometry of the pointing arrow connecting `xytext` to `xy`.

### Scenario 5: Ratings Distribution - `sns.boxplot()` vs `sns.violinplot()`
**1. Recommendation:** **`sns.boxplot()`** is recommended for this specific dataset.
**2. Two Data-Specific Reasons:**
- **Discrete 1-5 Rating Scale:** The rating scale is discrete (integers 1 to 5). Violin plots rely on continuous Kernel Density Estimation (KDE), which artificially smooths discrete points and generates misleading non-zero density tails below 1.0 and above 5.0 where no ratings exist. Boxplots cleanly show the exact discrete 25th, 50th, and 75th percentiles.
- **Explicit Outlier Detection:** The business requirement explicitly asks *'whether any category produces more extreme outliers'*. Box plots visually identify outliers as individual isolated points beyond the $1.5 \times \text{IQR}$ whiskers. Violin plots conceal individual outlier data points inside smooth density hulls.
**3. When Violin Plot is Preferable:**
- A violin plot is superior when analyzing **continuous, multi-modal distributions** with large sample sizes (e.g., bimodal customer delivery times or order spending amounts), where a box plot masks the dual-peak structure under a single interquartile box.

### Scenario 6: Seaborn Correlation Heatmap & Multicollinearity
**1. Generation & Interpretation:**
- Generated via `sns.heatmap(df.corr(), annot=True, cmap='coolwarm', vmin=-1, vmax=1)`.
- **Color Scale:** Represents Pearson correlation coefficients ranging from $-1.0$ (strong inverse correlation, e.g., deep blue) through $0.0$ (no linear correlation, neutral white) to $+1.0$ (perfect positive correlation, deep red).
- **Reading a Cell:** The intersection of row $X$ and column $Y$ indicates the pairwise Pearson correlation $r_{xy}$. Values close to $\pm 1.0$ show strong linear dependency.
**2. Multicollinearity Threshold & Convention:**
- A threshold of **$|r| \ge 0.70$ (or $0.80$)** is conventionally used to flag severe multicollinearity.
- **Why this threshold:** In linear regression, when $|r| > 0.70$, the variance inflation factor (VIF) increases significantly ($R^2 > 0.50$), inflating standard errors of regression coefficients, causing unstable coefficient signs, and degrading model interpretability.


---
## SECTION B: PRACTICAL CODING TASKS
### Task 1: Delivery Fee Array Calculator (NumPy)

In [ ]:
"""
M10-A1: Section B - Task 1: Delivery Fee Array Calculator (NumPy)
Stack: Python 3, NumPy
"""
import numpy as np

def main():
    # 1. Generate 1D array of 25 random delivery distances between 1.0 km and 15.0 km (seed=42)
    np.random.seed(42)
    distances = np.random.uniform(1.0, 15.0, size=25)

    # 2. Vectorised delivery fee calculation: fee = Rs 20 + Rs 5 * distance
    fees = 20.0 + 5.0 * distances

    print("=" * 60)
    print("TASK 1: NUMPY DELIVERY FEE ARRAY CALCULATOR")
    print("=" * 60)
    print(f"Total Orders Generated: {len(distances)}")
    print(f"Distances (km):\n{np.round(distances, 2)}")
    print(f"Delivery Fees (Rs):\n{np.round(fees, 2)}")
    print("-" * 60)

    # 3. Boolean indexing: orders where fee > Rs 60
    mask = fees > 60.0
    qualifying_distances = distances[mask]
    qualifying_fees = fees[mask]

    print(f"Orders with Delivery Fee > Rs 60 (Total: {len(qualifying_fees)} orders):")
    for idx, (d, f) in enumerate(zip(qualifying_distances, qualifying_fees), start=1):
        print(f"  [{idx}] Distance: {d:5.2f} km  -->  Fee: Rs {f:6.2f}")
    print("-" * 60)

    # 4. Statistical summary of complete fee array
    fee_min = np.min(fees)
    fee_max = np.max(fees)
    fee_mean = np.mean(fees)
    fee_std = np.std(fees)

    print("Delivery Fee Statistics (Rs):")
    print(f"  Minimum Fee : Rs {fee_min:6.2f}")
    print(f"  Maximum Fee : Rs {fee_max:6.2f}")
    print(f"  Mean Fee    : Rs {fee_mean:6.2f}")
    print(f"  Std Dev     : Rs {fee_std:6.2f}")
    print("=" * 60)

if __name__ == "__main__":
    main()



### Task 2: Restaurant Performance Analyser (Pandas)

In [ ]:
"""
M10-A1: Section B - Task 2: Restaurant Performance Analyser (Pandas)
Stack: Python 3, Pandas, NumPy
"""
import pandas as pd
import numpy as np

def main():
    np.random.seed(42)
    n_rows = 45

    restaurants = ['Spice Route', 'Biryani Blues', 'Urban Tadka', 'Wok Express', 'Sweet Cravings']
    cities = ['Mumbai', 'Delhi', 'Bengaluru']
    cuisines = ['Indian', 'Chinese', 'Desserts']

    data = {
        'order_id': [1000 + i for i in range(n_rows)],
        'restaurant_name': np.random.choice(restaurants, size=n_rows),
        'city': np.random.choice(cities, size=n_rows),
        'order_value': np.round(np.random.uniform(150.0, 950.0, size=n_rows), 2),
        'delivery_time_mins': np.random.randint(18, 55, size=n_rows),
        'rating': np.round(np.random.uniform(2.8, 5.0, size=n_rows), 1),
        'cuisine_type': np.random.choice(cuisines, size=n_rows)
    }

    df = pd.DataFrame(data)
    print("=" * 75)
    print("TASK 2: PANDAS RESTAURANT PERFORMANCE ANALYSER")
    print("=" * 75)
    print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
    print("Sample Rows:")
    print(df.head(4).to_string(index=False))
    print("-" * 75)

    # Single method chain: groupby() + agg()
    # Compute: mean order value, mean delivery time, and mean rating grouped by restaurant_name
    grouped_summary = (
        df.groupby('restaurant_name')
        .agg(
            mean_order_value=('order_value', 'mean'),
            mean_delivery_time=('delivery_time_mins', 'mean'),
            mean_rating=('rating', 'mean')
        )
        .round(2)
    )

    print("Grouped Summary by Restaurant:")
    print(grouped_summary.to_string())
    print("-" * 75)

    # Filter: mean rating > 4.0 and mean delivery time < 35 mins
    # Sort by mean_order_value descending, reset index
    final_ranked = (
        grouped_summary[
            (grouped_summary['mean_rating'] > 4.0) &
            (grouped_summary['mean_delivery_time'] < 35.0)
        ]
        .sort_values(by='mean_order_value', ascending=False)
        .reset_index()
    )

    print("Filtered & Ranked Output (Rating > 4.0 & Delivery Time < 35 mins):")
    if not final_ranked.empty:
        print(final_ranked.to_string(index=False))
    else:
        print("No restaurant met both strict thresholds in this synthetic sample.")
    print("=" * 75)

if __name__ == "__main__":
    main()



### Task 3: Monthly Order Trend Dashboard (Matplotlib)

In [ ]:
"""
M10-A1: Section B - Task 3: Monthly Order Trend Dashboard (Matplotlib)
Stack: Python 3, NumPy, Matplotlib
"""
import numpy as np
import matplotlib.pyplot as plt

def main():
    np.random.seed(42)
    months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
              'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    
    # 1. Synthetic monthly data using NumPy
    total_orders = np.random.randint(1000, 5000, size=12)
    avg_order_value = np.random.uniform(200, 400, size=12)
    monthly_revenue = total_orders * avg_order_value
    delivery_times_sample = np.random.normal(loc=28.0, scale=4.0, size=500)

    # Create figure with 3 subplots (1 row x 3 cols or 3 rows x 1 col; 3x1 stacked provides ample width for timeline)
    fig, axes = plt.subplots(3, 1, figsize=(11, 13))
    fig.suptitle('Food Delivery Platform: Monthly Performance Report', fontsize=16, fontweight='bold', y=0.99)

    # Subplot 1: Line chart of total orders with markers & annotations
    ax1 = axes[0]
    ax1.plot(months, total_orders, marker='o', color='#1f77b4', linewidth=2.5, markersize=8)
    ax1.set_title('Subplot 1: Total Orders Placed per Month', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Month', fontsize=11)
    ax1.set_ylabel('Total Orders', fontsize=11)
    ax1.grid(True, linestyle='--', alpha=0.5)

    for i, (m, val) in enumerate(zip(months, total_orders)):
        ax1.annotate(
            f"{val:,}",
            (m, val),
            textcoords="offset points",
            xytext=(0, 8),
            ha='center',
            fontsize=9,
            fontweight='bold',
            color='#0d47a1'
        )
    ax1.set_ylim(min(total_orders) - 500, max(total_orders) + 700)

    # Subplot 2: Bar chart of monthly revenue with conditional coloring & Rs 8,00,000 threshold
    ax2 = axes[1]
    threshold = 800000.0
    bar_colors = ['#2ca02c' if rev > threshold else '#d62728' for rev in monthly_revenue]
    bars = ax2.bar(months, monthly_revenue, color=bar_colors, edgecolor='black', alpha=0.85, width=0.6)
    ax2.axhline(threshold, color='black', linestyle='--', linewidth=1.8, label=f'Threshold (Rs 8,00,000)')
    ax2.set_title('Subplot 2: Monthly Revenue (Green > Rs 8L, Red <= Rs 8L)', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Month', fontsize=11)
    ax2.set_ylabel('Revenue (Rs)', fontsize=11)
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'Rs {x/1e5:.1f}L'))
    ax2.legend(loc='upper left', frameon=True)
    ax2.grid(True, axis='y', linestyle='--', alpha=0.5)

    # Subplot 3: Histogram of 500 simulated delivery times with 15 bins & Mean line
    ax3 = axes[2]
    sample_mean = np.mean(delivery_times_sample)
    ax3.hist(delivery_times_sample, bins=15, color='#9467bd', edgecolor='black', alpha=0.75)
    ax3.axvline(sample_mean, color='#b71c1c', linestyle='--', linewidth=2.2, label=f'Mean: {sample_mean:.2f} mins')
    ax3.set_title('Subplot 3: Distribution of Delivery Times (500 Orders, 15 Bins)', fontsize=12, fontweight='bold')
    ax3.set_xlabel('Delivery Time (minutes)', fontsize=11)
    ax3.set_ylabel('Frequency', fontsize=11)
    ax3.legend(loc='upper right', frameon=True)
    ax3.grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    output_png = 'food_delivery_dashboard.png'
    plt.savefig(output_png, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Task 3 complete. Dashboard saved to: {output_png} (DPI=150)")

if __name__ == "__main__":
    main()



### Task 4: Full EDA Pipeline - Food Delivery Insights (NumPy, Pandas, Matplotlib, Seaborn)

In [ ]:
"""
M10-A1: Section B - Task 4: Full EDA Pipeline - Food Delivery Insights
Stack: Python 3, NumPy, Pandas, Matplotlib, Seaborn
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def main():
    # 1. Generate 200-row dataset with seed=7
    np.random.seed(7)
    n = 200

    order_value = np.random.uniform(100.0, 800.0, size=n)
    distance_km = np.random.uniform(1.0, 20.0, size=n)
    delivery_time_mins = np.random.normal(loc=30.0, scale=8.0, size=n)
    # Ensure realistic non-negative delivery time
    delivery_time_mins = np.clip(delivery_time_mins, 10.0, 60.0)
    rating = np.random.uniform(1.0, 5.0, size=n)
    discount_pct = np.random.uniform(0.0, 30.0, size=n)

    df = pd.DataFrame({
        'order_value': order_value,
        'distance_km': distance_km,
        'delivery_time_mins': delivery_time_mins,
        'rating': rating,
        'discount_pct': discount_pct
    })

    print("=" * 70)
    print("TASK 4: FULL EDA PIPELINE")
    print("=" * 70)
    print(f"Initial shape: {df.shape}")

    # 2. Artificially introduce 5% null values into delivery_time_mins and rating
    null_count = int(0.05 * n)
    null_idx_time = np.random.choice(n, size=null_count, replace=False)
    null_idx_rating = np.random.choice(n, size=null_count, replace=False)

    df.loc[null_idx_time, 'delivery_time_mins'] = np.nan
    df.loc[null_idx_rating, 'rating'] = np.nan
    print(f"Nulls introduced -> delivery_time_mins: {df['delivery_time_mins'].isna().sum()}, rating: {df['rating'].isna().sum()}")

    # Impute nulls with column medians
    median_time = df['delivery_time_mins'].median()
    median_rating = df['rating'].median()
    df['delivery_time_mins'].fillna(median_time, inplace=True)
    df['rating'].fillna(median_rating, inplace=True)
    print(f"Nulls after median imputation -> delivery_time_mins: {df['delivery_time_mins'].isna().sum()}, rating: {df['rating'].isna().sum()}")
    print("-" * 70)

    # 3. Add derived column: delivery_speed_kmph = distance_km / (delivery_time_mins / 60)
    df['delivery_speed_kmph'] = df['distance_km'] / (df['delivery_time_mins'] / 60.0)

    # Classify into speed_band ('Slow', 'Normal', 'Fast') using pd.qcut() with 3 equal-frequency bins
    df['speed_band'] = pd.qcut(df['delivery_speed_kmph'], q=3, labels=['Slow', 'Normal', 'Fast'])
    print("Speed band distribution:")
    print(df['speed_band'].value_counts())
    print("-" * 70)

    # 4. Correlation heatmap of numeric columns with annot=True and diverging palette
    numeric_cols = ['order_value', 'distance_km', 'delivery_time_mins', 'rating', 'discount_pct', 'delivery_speed_kmph']
    corr_matrix = df[numeric_cols].corr()

    plt.figure(figsize=(8, 6.5))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, center=0, square=True, linewidths=0.5)
    plt.title('Food Delivery Dataset: Correlation Heatmap', fontsize=13, fontweight='bold', pad=12)
    heatmap_file = 'correlation_heatmap.png'
    plt.savefig(heatmap_file, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Heatmap saved to: {heatmap_file} (DPI=150)")

    # 5. Seaborn pairplot coloured by speed_band
    pairplot_cols = ['order_value', 'distance_km', 'delivery_time_mins', 'rating', 'speed_band']
    g = sns.pairplot(df[pairplot_cols], hue='speed_band', palette='Set1', diag_kind='kde')
    g.fig.suptitle('Food Delivery Metrics Pairplot by Speed Band', y=1.02, fontsize=14, fontweight='bold')
    pairplot_file = 'pairplot.png'
    g.savefig(pairplot_file, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Pairplot saved to: {pairplot_file} (DPI=150)")
    print("=" * 70)

if __name__ == "__main__":
    main()



---
## SECTION C: MINI CAPSTONE PROJECT
### Food Delivery Analytics Console - Full EDA Report

In [ ]:
"""
M10-A1: Section C - Mini Capstone Project
Food Delivery Analytics Console - Full EDA Report
Stack: Python 3, NumPy, Pandas, Matplotlib, Seaborn
"""
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def generate_and_clean_dataset(n_rows=250, seed=42):
    """Generates synthetic dataset at startup, introduces nulls, and cleans numeric columns."""
    np.random.seed(seed)
    
    restaurants = ['Royal Spice', 'Punjab Grill', 'Dragon Wok', 'Cafe Mocha', 'Burger Barn', 'Tandoor Treats']
    cities = ['Mumbai', 'Delhi', 'Bengaluru', 'Hyderabad']
    cuisines = ['North Indian', 'Chinese', 'Fast Food', 'Beverages', 'Mughlai']

    order_value = np.random.uniform(120.0, 1100.0, size=n_rows)
    distance_km = np.random.uniform(1.2, 22.0, size=n_rows)
    delivery_time_mins = np.random.normal(loc=32.0, scale=7.5, size=n_rows)
    delivery_time_mins = np.clip(delivery_time_mins, 12.0, 70.0)
    rating = np.random.uniform(1.5, 5.0, size=n_rows)
    discount_pct = np.random.uniform(0.0, 35.0, size=n_rows)
    tip_amount = np.random.choice([0, 10, 20, 30, 50, 100], size=n_rows, p=[0.4, 0.25, 0.15, 0.1, 0.07, 0.03])

    df = pd.DataFrame({
        'order_id': [50000 + i for i in range(n_rows)],
        'restaurant_name': np.random.choice(restaurants, size=n_rows),
        'city': np.random.choice(cities, size=n_rows),
        'cuisine': np.random.choice(cuisines, size=n_rows),
        'order_value': np.round(order_value, 2),
        'distance_km': np.round(distance_km, 2),
        'delivery_time_mins': np.round(delivery_time_mins, 1),
        'rating': np.round(rating, 1),
        'discount_pct': np.round(discount_pct, 1),
        'tip_amount': tip_amount
    })

    # Artificially inject nulls in numeric columns to test data cleaning requirement
    null_idx = np.random.choice(n_rows, size=int(0.04 * n_rows), replace=False)
    df.loc[null_idx, 'delivery_time_mins'] = np.nan
    df.loc[null_idx, 'rating'] = np.nan

    # Clean / Impute nulls with medians
    for col in ['order_value', 'distance_km', 'delivery_time_mins', 'rating', 'discount_pct', 'tip_amount']:
        if df[col].isna().sum() > 0:
            median_val = df[col].median()
            df[col].fillna(median_val, inplace=True)

    return df

def option_1_summary_statistics(df):
    """
    Primary Technique: NumPy statistical functions
    Calculates and displays comprehensive statistical metrics.
    """
    print("\n" + "=" * 60)
    print("OPTION 1: SUMMARY STATISTICS (NUMPY VECTOR COMPUTATIONS)")
    print("=" * 60)
    numeric_cols = ['order_value', 'distance_km', 'delivery_time_mins', 'rating', 'discount_pct', 'tip_amount']
    
    header = f"{'Metric Column':<20}{'Min':<10}{'Max':<10}{'Mean':<12}{'Median':<12}{'Std Dev':<10}"
    print(header)
    print("-" * 74)
    for col in numeric_cols:
        arr = df[col].to_numpy()
        c_min = np.min(arr)
        c_max = np.max(arr)
        c_mean = np.mean(arr)
        c_median = np.median(arr)
        c_std = np.std(arr)
        print(f"{col:<20}{c_min:<10.2f}{c_max:<10.2f}{c_mean:<12.2f}{c_median:<12.2f}{c_std:<10.2f}")
    print("=" * 60 + "\n")

def option_2_distribution_analysis(df):
    """
    Primary Technique: Matplotlib multi-panel subplots
    Saves distribution analysis non-interactively to PNG at DPI >= 150.
    """
    print("\n" + "=" * 60)
    print("OPTION 2: DISTRIBUTION ANALYSIS (MATPLOTLIB SUBPLOTS)")
    print("=" * 60)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle('Food Delivery Analytics: Order Value & Delivery Time Distributions', fontsize=14, fontweight='bold')

    # Subplot A: Order value histogram
    axes[0].hist(df['order_value'], bins=18, color='#17becf', edgecolor='black', alpha=0.8)
    axes[0].axvline(df['order_value'].mean(), color='red', linestyle='--', linewidth=1.8, label=f"Mean: Rs {df['order_value'].mean():.1f}")
    axes[0].set_title('Order Value Distribution', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Order Value (Rs)')
    axes[0].set_ylabel('Order Count')
    axes[0].legend()
    axes[0].grid(True, linestyle=':', alpha=0.6)

    # Subplot B: Delivery time boxplot
    axes[1].boxplot(df['delivery_time_mins'], vert=True, patch_artist=True,
                    boxprops=dict(facecolor='#aec7e8', color='navy'),
                    medianprops=dict(color='red', linewidth=2))
    axes[1].set_title('Delivery Time Distribution & Outliers', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Delivery Time (minutes)')
    axes[1].grid(True, linestyle=':', alpha=0.6)

    plt.tight_layout()
    chart_filename = 'capstone_distribution_analysis.png'
    plt.savefig(chart_filename, dpi=180, bbox_inches='tight')
    plt.close()
    print(f"Distribution chart successfully generated and saved to '{chart_filename}' (DPI=180).\n")

def option_3_correlation_heatmap(df):
    """
    Primary Technique: Seaborn statistical heatmap
    Saves correlation matrix non-interactively to PNG at DPI >= 150.
    """
    print("\n" + "=" * 60)
    print("OPTION 3: CORRELATION HEATMAP (SEABORN HEATMAP)")
    print("=" * 60)
    numeric_cols = ['order_value', 'distance_km', 'delivery_time_mins', 'rating', 'discount_pct', 'tip_amount']
    corr = df[numeric_cols].corr()

    plt.figure(figsize=(8, 6.5))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='Spectral', vmin=-1, vmax=1, center=0, square=True, linewidths=0.5)
    plt.title('Numeric Feature Correlation Heatmap', fontsize=13, fontweight='bold', pad=12)
    
    chart_filename = 'capstone_correlation_heatmap.png'
    plt.savefig(chart_filename, dpi=180, bbox_inches='tight')
    plt.close()
    print(f"Heatmap successfully generated and saved to '{chart_filename}' (DPI=180).\n")

def option_4_restaurant_performance_report(df):
    """
    Primary Technique: Pandas groupby aggregation & ranking
    Aggregates metrics by restaurant brand.
    """
    print("\n" + "=" * 60)
    print("OPTION 4: RESTAURANT PERFORMANCE REPORT (PANDAS GROUPBY)")
    print("=" * 60)
    
    perf = (
        df.groupby('restaurant_name')
        .agg(
            total_orders=('order_id', 'count'),
            mean_order_value=('order_value', 'mean'),
            mean_delivery_time=('delivery_time_mins', 'mean'),
            mean_rating=('rating', 'mean')
        )
        .round(2)
        .sort_values(by='mean_rating', ascending=False)
        .reset_index()
    )
    print(perf.to_string(index=False))
    print("=" * 60 + "\n")

def print_final_summary_report(df):
    """Prints plain-text summary report required upon console exit."""
    print("\n" + "#" * 65)
    print("FOOD DELIVERY ANALYTICS - FINAL CONSOLE SUMMARY REPORT")
    print("#" * 65)

    # 1. Top 3 restaurants by mean rating
    top_3 = (
        df.groupby('restaurant_name')['rating']
        .mean()
        .round(2)
        .sort_values(ascending=False)
        .head(3)
        .reset_index()
    )
    print("1. TOP 3 RESTAURANTS BY MEAN RATING:")
    for rank, row in top_3.iterrows():
        print(f"   Rank {rank+1}: {row['restaurant_name']:<20} (Mean Rating: {row['rating']:.2f} / 5.0)")

    # 2. Numeric column pair with the highest absolute Pearson correlation
    numeric_cols = ['order_value', 'distance_km', 'delivery_time_mins', 'rating', 'discount_pct', 'tip_amount']
    corr_matrix = df[numeric_cols].corr()
    
    # Mask diagonal and duplicate lower triangle
    upper_corr = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    corr_unstacked = upper_corr.unstack().dropna()
    abs_sorted = corr_unstacked.abs().sort_values(ascending=False)
    
    highest_pair = abs_sorted.index[0]  # (col2, col1)
    highest_corr_val = corr_matrix.loc[highest_pair[0], highest_pair[1]]
    
    print("\n2. HIGHEST ABSOLUTE PEARSON CORRELATION PAIR:")
    print(f"   Pair        : '{highest_pair[0]}' and '{highest_pair[1]}'")
    print(f"   Correlation : {highest_corr_val:.4f} (Absolute: {abs(highest_corr_val):.4f})")

    # 3. Overall mean and standard deviation of delivery time across full dataset
    time_mean = np.mean(df['delivery_time_mins'].to_numpy())
    time_std = np.std(df['delivery_time_mins'].to_numpy())
    print("\n3. OVERALL DELIVERY TIME METRICS:")
    print(f"   Mean Delivery Time    : {time_mean:.2f} minutes")
    print(f"   Std Dev Delivery Time : {time_std:.2f} minutes")
    print("#" * 65 + "\n")

def main():
    df = generate_and_clean_dataset(n_rows=250, seed=42)
    
    menu_text = (
        "\n" + "=" * 45 + "\n"
        "   FOOD DELIVERY ANALYTICS CONSOLE\n"
        "=" * 45 + "\n"
        "1. Summary Statistics (NumPy Operations)\n"
        "2. Distribution Analysis (Matplotlib Subplots)\n"
        "3. Correlation Heatmap (Seaborn Heatmap)\n"
        "4. Restaurant Performance Report (Pandas GroupBy)\n"
        "5. Exit & Generate Executive Summary\n"
        "-" * 45 + "\n"
        "Enter your choice (1-5): "
    )

    while True:
        try:
            choice = input(menu_text).strip()
            if choice == '1':
                option_1_summary_statistics(df)
            elif choice == '2':
                option_2_distribution_analysis(df)
            elif choice == '3':
                option_3_correlation_heatmap(df)
            elif choice == '4':
                option_4_restaurant_performance_report(df)
            elif choice == '5':
                print("\nExiting console application...")
                print_final_summary_report(df)
                break
            else:
                print("Invalid selection. Please choose an option from 1 to 5.")
        except (KeyboardInterrupt, EOFError):
            print("\nApplication terminated.")
            print_final_summary_report(df)
            break

if __name__ == "__main__":
    main()



---
## SECTION D: AI-AUGMENTED LEARNING
### Prompts, Original AI Code, Human Debugging, and Fixes

In [ ]:
"""
M10-A1: Section D - AI-Augmented Learning
Contains both AI's original code and the human-corrected robust production code.
"""

# ==============================================================================
# 1. THE EXACT PROMPT GIVEN TO THE AI TOOL:
# ==============================================================================
"""
PROMPT:
"Write a complete Python script using NumPy, Pandas, Matplotlib, and Seaborn for a food delivery dataset.
Requirements:
1. Generate a synthetic dataset with at least 5 numeric columns (order_value, distance_km, delivery_time,
   rating, discount_pct) and one categorical column (cuisine_type) using NumPy and load into Pandas DataFrame.
2. Produce a Seaborn pairplot of all numeric columns, coloured by cuisine_type, and save as a PNG file.
3. Compute the Pearson correlation matrix and display it as a Seaborn heatmap with annotated cell values
   and a diverging color palette.
4. Export the heatmap as a separate PNG file at DPI 200 with a descriptive filename."
"""

# ==============================================================================
# 2A. THE AI'S ORIGINAL GENERATED CODE (WITH BUGS / LIMITATIONS):
# ==============================================================================
def ai_original_code():
    import numpy as np
    import pandas as pd
    import seaborn as sns
    import matplotlib.pyplot as plt

    # AI Bug 1: Pairplot fails when categorical column has NaN/mixed types or object dtype issues.
    # AI Bug 2: The AI computed correlation on the whole DataFrame without numeric_only=True, which raises TypeError in modern Pandas (>= 2.0).
    # AI Bug 3: Pairplot savefig was missing explicit DPI=200 or DPI parameter entirely.
    # AI Bug 4: Did not close plt figures, causing memory leakage and canvas overplotting between figures.
    
    np.random.seed(10)
    data = {
        'order_value': np.random.uniform(100, 1000, 100),
        'distance_km': np.random.uniform(1, 20, 100),
        'delivery_time': np.random.normal(30, 10, 100),
        'rating': np.random.uniform(1, 5, 100),
        'discount_pct': np.random.uniform(0, 40, 100),
        'cuisine_type': np.random.choice(['Indian', 'Chinese', 'Italian', None], 100) # AI introduced None without handling
    }
    df = pd.DataFrame(data)

    # Broken in Pandas 2.0+ without numeric_only=True, and breaks pairplot with None values
    # pairplot = sns.pairplot(df, hue='cuisine_type')
    # pairplot.savefig("ai_pairplot.png")  # Missing explicit DPI requirement
    # corr = df.corr() # Crashes in Pandas 2.0 due to string/None column
    # sns.heatmap(corr, annot=True, cmap='vlag')
    # plt.savefig('heatmap_ai.png', dpi=200)
    pass

# ==============================================================================
# 2B. HUMAN-CORRECTED AND ROBUST CODE (PRODUCES SUBMISSION ARTIFACTS):
# ==============================================================================
def human_corrected_code():
    import numpy as np
    import pandas as pd
    import seaborn as sns
    import matplotlib.pyplot as plt

    np.random.seed(10)
    n = 150
    data = {
        'order_value': np.round(np.random.uniform(100, 1000, n), 2),
        'distance_km': np.round(np.random.uniform(1, 20, n), 2),
        'delivery_time': np.round(np.random.normal(30, 8, n), 1),
        'rating': np.round(np.random.uniform(1, 5, n), 1),
        'discount_pct': np.round(np.random.uniform(0, 35, n), 1),
        'cuisine_type': np.random.choice(['Indian', 'Chinese', 'Italian', 'Continental'], n)
    }
    df = pd.DataFrame(data)

    # FIX 1: Clean category dtypes and ensure no NaNs that break Seaborn KDE
    df['cuisine_type'] = df['cuisine_type'].astype('category')

    # FIX 2: Generate pairplot with explicit DPI >= 200, clean layout, and close figure
    pp = sns.pairplot(df, hue='cuisine_type', palette='Set2', diag_kind='kde')
    pp.fig.suptitle('Food Delivery Metrics Pairplot by Cuisine', y=1.02, fontsize=13, fontweight='bold')
    pp_file = 'corrected_cuisine_pairplot.png'
    pp.savefig(pp_file, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"[Section D] Corrected pairplot saved to '{pp_file}' (DPI=200)")

    # FIX 3: Enforce numeric_only=True for Pandas 2.0 compatibility
    numeric_corr = df.select_dtypes(include=[np.number]).corr()

    # FIX 4: Properly configure heatmap with diverging palette, annotations, and export at DPI 200
    plt.figure(figsize=(7.5, 6))
    sns.heatmap(numeric_corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, center=0, square=True, linewidths=0.5)
    plt.title('Food Delivery Feature Correlation Matrix', fontsize=12, fontweight='bold', pad=12)
    heatmap_file = 'corrected_correlation_heatmap_200dpi.png'
    plt.savefig(heatmap_file, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"[Section D] Corrected heatmap saved to '{heatmap_file}' (DPI=200)")

# ==============================================================================
# 3. NOTE EXPLAINING FIXES (REQUIRED 3-4 LINES):
# ==============================================================================
"""
EXPLANATION NOTE:
The AI tool generated code that called `df.corr()` directly across the full DataFrame without 
specifying `numeric_only=True`, which triggers a critical TypeError in Pandas 2.0+ due to the 
presence of the categorical column. Furthermore, the AI omitted DPI specifications on the pairplot 
export and failed to handle potential nulls/category typing in `cuisine_type`, which causes KDE 
sub-diagonals to crash. I explicitly isolated numeric dtypes for correlation, sanitized the 
categorical series, added `plt.close()` to prevent canvas memory leaks, and enforced `dpi=200` 
on all export routines.
"""

if __name__ == "__main__":
    human_corrected_code()

